# A decision tree using Shannon entropy and information gain

**Status:** ✅ Runs end to end — carries a regression test for the `and col` bug that disabled feature 0

**Demonstrates:** Information theory · greedy splitting · minimum leaf size. Superseded by `ml_from_scratch.decision_tree`.

---

> Part of a portfolio of applied ML and statistics work. Every notebook
> here is executed end to end; the bug record is in `BUGFIXES.md`.


In [ ]:

import numpy as np
from matplotlib import pyplot as plt

rng = np.random.default_rng(23)


## What was broken

1. `choose_best_feature(data)` referenced an undefined name `data_n` instead of
   its own `data` argument, so every call raised `NameError`.
2. **The split guard read `if ent < min_avg_ent and col:`.** Since column `0`
   is falsy in Python, `and col` was False whenever the first feature was the
   best candidate - so **feature 0 could never be selected**. No error, no
   warning, just a permanently worse model. The guard is gone, and a test
   below pins the behaviour.
3. `predict` looked up `self.labels`, which was never populated.

The same code, corrected and tested, also lives in
`src/ml_from_scratch/decision_tree.py`.

## 1. Shannon entropy

In [ ]:

def shannon_entropy(labels):
    """Compute Shannon entropy in bits for a set of class labels.

    Args:
        labels: Array of labels of any type. Only the value counts matter.

    Returns:
        float: Entropy in ``[0, log2(k)]`` where ``k`` is the number of
        distinct classes. Zero for an empty input.

    Example:
        >>> shannon_entropy([0, 0, 1, 1])
        1.0
        >>> shannon_entropy([1, 1, 1])
        0.0
    """
    labels = np.asarray(labels)
    if labels.size == 0:
        return 0.0
    _, counts = np.unique(labels, return_counts=True)
    probabilities = counts / counts.sum()
    probabilities = probabilities[probabilities > 0]      # guards log2(0)
    return float(-np.sum(probabilities * np.log2(probabilities)))


## 2. Information gain

In [ ]:

def information_gain(parent, left, right):
    """Compute the information gain of a binary split.

    Args:
        parent: All labels at the parent node.
        left: Labels routed to the left child.
        right: Labels routed to the right child.

    Returns:
        float: Reduction in entropy, in bits. Non-negative for a valid split.

    Example:
        >>> round(information_gain([0, 0, 1, 1], [0, 0], [1, 1]), 6)
        1.0
    """
    parent = np.asarray(parent)
    n_parent = parent.size
    if n_parent == 0:
        return 0.0
    n_left = np.asarray(left).size
    n_right = np.asarray(right).size
    remainder = (
        n_left * shannon_entropy(left) + n_right * shannon_entropy(right)
    ) / n_parent
    return shannon_entropy(parent) - remainder


## 3. Finding the best split

The BUGFIX that matters in this cell: the original guard was
`if ent < min_avg_ent and col:`. Because `0` is falsy, column 0 could never
win. The loop below has **no such clause** - every feature is a candidate.

In [ ]:

def choose_best_feature(data):
    """Search all features and midpoints for the highest-gain split.

    BUGFIX: the original referenced an undefined name ``data_n`` instead of
    ``data``, and guarded its best-split update with ``and col``. Since column
    0 is falsy, that clause made feature 0 permanently ineligible.

    Args:
        data: Array of shape ``(n_samples, n_features + 1)`` where column 0
            holds the label and the rest hold the features.

    Returns:
        tuple: ``(feature_index, threshold, gain)`` for the best split, or
        ``None`` when no split yields positive gain.
    """
    labels = data[:, 0]
    best_feature, best_threshold, best_gain = None, None, 0.0

    for column in range(1, data.shape[1]):
        values = np.unique(data[:, column])
        if values.size < 2:
            continue

        # A binary split can only change behaviour between adjacent distinct
        # values, so the midpoints are the only candidate thresholds.
        for threshold in (values[:-1] + values[1:]) / 2.0:
            mask = data[:, column] <= threshold
            gain = information_gain(labels, labels[mask], labels[~mask])
            if gain > best_gain + 1e-12:
                best_feature, best_threshold, best_gain = (
                    column - 1,                       # 0-based feature index
                    float(threshold),
                    float(gain),
                )

    if best_feature is None:
        return None
    return best_feature, best_threshold, best_gain


## 4. The tree itself

In [ ]:

class DecisionTree:
    """Binary CART classifier using entropy-based information gain.

    Stops splitting at a node when it is pure, smaller than
    ``min_samples_split``, would violate ``min_samples_leaf``, exceeds
    ``max_depth``, or when no split yields positive gain.

    Attributes:
        root_ (tuple | scalar): Fitted tree. Internal nodes are
            ``(feature, threshold, left, right)``; leaves hold a class label.
        depth_ (int): Depth of the fitted tree.
    """

    def __init__(self, min_samples_leaf=1, min_samples_split=2, max_depth=None):
        """Configure the tree.

        Args:
            min_samples_leaf: Smallest permitted leaf size.
            min_samples_split: Smallest node size that may split.
            max_depth: Maximum depth, or ``None`` for no limit.
        """
        self.min_samples_leaf = min_samples_leaf
        self.min_samples_split = min_samples_split
        self.max_depth = max_depth
        self.root_ = None
        self.depth_ = 0

    def fit(self, x, y):
        """Fit the tree.

        BUGFIX: the original ``predict`` read ``self.labels``, a dictionary that
        was never created. Prediction now walks the stored tree.

        Args:
            x: Features of shape ``(n_samples, n_features)``.
            y: Labels of shape ``(n_samples,)``.

        Returns:
            DecisionTree: self, for chaining.
        """
        data = np.column_stack([np.asarray(y), np.asarray(x, dtype=float)])
        self.depth_ = 0
        self.root_ = self._build(data, 0)
        return self

    def _build(self, data, depth):
        """Recursively build the subtree rooted at ``data``.

        Args:
            data: Array whose first column is the label.
            depth: Current depth; the root is ``0``.

        Returns:
            tuple: A ``(feature, threshold, left, right)`` node, or a scalar
            class label for a leaf.
        """
        self.depth_ = max(self.depth_, depth)
        labels = data[:, 0]
        values, counts = np.unique(labels, return_counts=True)
        majority = values[np.argmax(counts)]

        if (
            shannon_entropy(labels) == 0.0
            or data.shape[0] < self.min_samples_split
            or data.shape[0] < 2 * self.min_samples_leaf
            or (self.max_depth is not None and depth >= self.max_depth)
        ):
            return majority

        split = choose_best_feature(data)
        if split is None:
            return majority

        feature, threshold, _ = split
        mask = data[:, feature + 1] <= threshold       # +1 for the label column

        # BUGFIX: the original passed data[~mask] as the LEFT child while
        # predict() routed samples satisfying `x <= threshold` to the left.
        # The two disagreed, so the tree learned the correct split and then
        # predicted its exact opposite - 0% accuracy on separable data, and
        # worse than chance on noisy data. The left child must be the samples
        # that satisfy the routing condition in predict().
        return (
            feature,
            threshold,
            self._build(data[mask], depth + 1),
            self._build(data[~mask], depth + 1),
        )

    def predict(self, x):
        """Predict labels by routing each sample down the tree.

        Args:
            x: Features of shape ``(n_samples, n_features)``.

        Returns:
            numpy.ndarray: Predicted labels of shape ``(n_samples,)``.

        Raises:
            RuntimeError: If called before :meth:`fit`.
            ValueError: If ``x`` is not two-dimensional. A 1-D array would
                iterate over scalars and index them as if they were feature
                vectors, producing a confusing ``IndexError`` further down.
        """
        if self.root_ is None:
            raise RuntimeError("DecisionTree has not been fitted yet; call fit first.")
        x = np.asarray(x, dtype=float)
        if x.ndim != 2:
            raise ValueError(
                f"x must be 2-D (n_samples, n_features), got shape {x.shape}. "
                "Use x.reshape(-1, 1) for a single feature."
            )

        def route(sample):
            node = self.root_
            while isinstance(node, tuple):
                feature, threshold, left, right = node
                node = left if sample[feature] <= threshold else right
            return node

        return np.array([route(row) for row in x])

    def score(self, x, y):
        """Return the fraction of correctly classified samples.

        Args:
            x: Features of shape ``(n_samples, n_features)``.
            y: True labels of shape ``(n_samples,)``.

        Returns:
            float: Accuracy in ``[0, 1]``.
        """
        return float(np.mean(self.predict(x) == np.asarray(y)))


## 5. The regression test for the `and col` bug

Feature 0 is the *only* informative column here. A tree that cannot select
column 0 scores at chance.

In [ ]:

# REGRESSION TEST for the `and col` bug: if feature 0 could not be selected,
# this tree would have to fall back to column 1 (pure noise) and score ~0.5.
# BUGFIX: this was 1-D (shape (120,)), so predict() iterated over scalars and
# raised IndexError. A single feature must still be passed as a column.
X_first_feature = np.array([0.0, 0.1, 0.2, 0.8, 0.9, 1.0] * 20).reshape(-1, 1)
y_first_feature = np.array([0, 0, 0, 1, 1, 1] * 20)

regression_tree = DecisionTree(min_samples_leaf=1).fit(X_first_feature, y_first_feature)
chosen = regression_tree.root_[0] if isinstance(regression_tree.root_, tuple) else None
print("feature selected by the root split:", chosen)
print("accuracy:", regression_tree.score(X_first_feature, y_first_feature))
print("PASS - feature 0 is usable" if chosen == 0
      else "FAIL - feature 0 was never eligible, the original bug is present")


## 6. Does `min_samples_leaf` actually regularise?

In [ ]:

def count_nodes(node):
    """Count every node in a fitted tree, leaves included.

    Args:
        node: A node from :attr:`DecisionTree.root_`.

    Returns:
        int: Total node count.
    """
    if not isinstance(node, tuple):
        return 1
    return 1 + count_nodes(node[2]) + count_nodes(node[3])


In [ ]:

n_samples = 400
X_noisy = rng.normal(size=(n_samples, 3))
y_noisy = (X_noisy[:, 0] + 0.5 * rng.normal(size=n_samples) > 0).astype(int)

print(f"{'min_samples_leaf':>17} {'nodes':>7} {'depth':>6} {'train acc':>10} {'test acc':>9}")
rng2 = np.random.default_rng(5)
split = rng2.permutation(n_samples)
X_fit, y_fit = X_noisy[split[:300]], y_noisy[split[:300]]
X_hold, y_hold = X_noisy[split[300:]], y_noisy[split[300:]]

for leaf in (1, 5, 20, 50):
    tree = DecisionTree(min_samples_leaf=leaf).fit(X_fit, y_fit)
    nodes = count_nodes(tree.root_)
    print(f"{leaf:17d} {nodes:7d} {tree.depth_:6d} "
          f"{tree.score(X_fit, y_fit):10.4f} {tree.score(X_hold, y_hold):9.4f}")
